In [1]:
# Import packages
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options

from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import ElementClickInterceptedException
from selenium.common.exceptions import ElementNotInteractableException

from bs4 import BeautifulSoup

import time, re, requests
import pandas as pd

In [2]:
# Open the browser without showing it on screen
opts = Options()
opts.add_argument("--headless=new")
opts.add_argument("window-size=2560,1600")

# Open the Chrome browser
driver = webdriver.Chrome() #(options=opts)

# Navigate to the Project Gutenberg homepage
url = "https://www.gutenberg.org/"
driver.get(url)

# Set the implicit waiting time to 0.5 seconds
driver.implicitly_wait(0.5)
# Set the explicit wwaiting time to 20 seconds
wait = WebDriverWait(driver, 20)

In [3]:
# Find the navigation bar links
nav_bar = driver.find_elements(By.CSS_SELECTOR, ".main-links a")
if nav_bar:
    for link in nav_bar:
        # Print the navigation bar links
        print(link.text, link.get_attribute("href"))

 https://www.gutenberg.org/browse/scores/top
 https://www.gutenberg.org/ebooks/categories
Reading Lists https://www.gutenberg.org/ebooks/bookshelf/
Search Options https://www.gutenberg.org/ebooks/


In [4]:
# Navigate to the Reading Lists
reading_lists = driver.find_element(By.CSS_SELECTOR, "a[class='link-reading-lists']")
reading_lists.click()

# Pause execution for 5 seconds
time.sleep(5)

In [5]:
# Navigate to the Main Categories webpage
main_categories = driver.find_element(By.CSS_SELECTOR, "div[class='page_content'] p a")
main_categories.click()

# Pause execution for 5 seconds
time.sleep(5)

In [6]:
# Navigate to the Science-Fiction & Fantasy webpage
sf_fantasy = driver.find_element(By.XPATH, "/html/body/div[1]/div/div/div[1]/ul/li[14]/a")
sf_fantasy.click()

time.sleep(5)

In [ ]:
def scrape_rows():
    """
    Scrape the rows from the table of a specific year.

    Returns:
        dict: Scraped table row values.
    """
    # Load Selenium's page_source into BeautifulSoup
    document = BeautifulSoup(driver.page_source, "html.parser")
    index = 2
    authors = []
    for book in document.select(".booklink"):
        if driver.find_element(By.XPATH, f'//*[@id="content"]/div/ul/li[{index}]/a/span[2]/span[2]'):
            authors.append(driver.find_element(By.XPATH, f'//*[@id="content"]/div/ul/li[{index}]/a/span[2]/span[2]'))
        else:
            authors.append("")
        
        index += 1

    return {
        "titles": document.select(".title"),
        "authors": authors,
        "downloads": document.select(".extra"),
        "texts": driver.find_elements(By.CSS_SELECTOR, ".booklink a")
    }

In [ ]:
def create_text():
    """
    Turn the scraped rows into text.

    Returns:
        dict: Table row values in text form.
    """
    # Assign scrape_rows() function to scrape_rows_result variable
    scrape_rows_result = scrape_rows()

    return {
        "title_text": [title.text for title in scrape_rows_result["titles"]],
        "author_text": [author for author in scrape_rows_result["authors"]],
        "download_text": [download.text for download in scrape_rows_result["downloads"]],
        "text_text": [text.get_attribute("href") for text in scrape_rows_result["texts"]]
    }

In [9]:
def collect_data():
    """
    Iterates over table rows and scrapes it's content.
    
    Returns:
        List: Non-emtpy table rows.
    """
    # Assign create_text() function to create_text_result variable
    create_text_result = create_text()
    # Count the amount of table rows
    row_count = len(driver.find_elements(By.CSS_SELECTOR, ".booklink"))

    # Append text to list
    specific_page = []
    row_index = 0
    for row in range(row_count):
        specific_page.append({
            "Title": create_text_result["title_text"][row_index],
            "Author": create_text_result["author_text"][row_index],
            "Downloads": create_text_result["download_text"][row_index],
            "Text": create_text_result["text_text"][row_index]
        })
        row_index += 1

    return specific_page

In [48]:
# Instantiate the df_novels DataFrame
df_novels = pd.DataFrame()

# Count the amount of books on the webpage
page_length = 169

# Iterate through the webpages
for page in range(4, 5):
    # Create the df_append DataFrame, based on the results of the scrape_page() function
    df_append = pd.DataFrame(collect_data())
    # Append the df_append DataFrame to the df_novels DataFrame
    df_novels = pd.concat([df_novels, df_append], ignore_index=True)

    try:
        driver.find_element(By.CSS_SELECTOR, "a[title='Go to the next page of results.']").click()
        time.sleep(3)
    except NoSuchElementException:
        pass

# Show the df_novels DataFrame
df_novels

TypeError: 'NoneType' object is not subscriptable